# Problem 2: Full-Document Stuffing vs. RAG — Evaluation Notebook

**Goal:** Build and evaluate two competing QA architectures over SEC 10-K filings.

## What this notebook does

1. Loads the cleaned SEC corpus from `/SEC_NLP/sec_corpus_clean/` (excluding stub files)
2. Sets up two QA pipelines:
   - **Challenger:** Full-Document Stuffing — concatenates all four sections (Item 1, 1A, 7, 8) with content-aware section localization → Qwen 2.5-7B generates answer
   - **Champion:** RAG (BGE-Small embed → Qdrant retrieval with ticker filter, top-5 chunks) → Qwen 2.5-7B generates answer
3. Runs both pipelines on a 16-question golden dataset (AAPL + MSFT × 4 sections × 3 question types)
4. Independently judges both with **GPT-4o** (relevance + faithfulness)
5. Computes Numerical Exact Match for factual_lookup questions
6. Reports side-by-side comparison: relevance, faithfulness, EM, latency, token cost

## Before you run

- Mount Google Drive (Cell below)
- Make sure `/MyDrive/SEC_NLP/sec_corpus_clean/` exists (or update `G_DRIVE_CORPUS_PATH`)
- Make sure `/MyDrive/SEC_NLP/section_recovery.py` exists (imported in Section 4)
- Provide an OpenAI API key via Colab Secrets (`OPENAI_API_KEY`)
- Use an A100 (or higher) GPU (faster, more memory storage) runtime (Runtime → Change runtime type → A100 GPU)


## 0. Setup: Drive mount + install libraries

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install required libraries (uncomment on first run)
!pip install -q transformers accelerate bitsandbytes sentence-transformers qdrant-client anthropic

print("Environment ready.")

In [ ]:
# ─── Paths (update these if your folder layout differs) ──────────────────────
G_DRIVE_BASE        = 'data/SEC_NLP'
G_DRIVE_CORPUS_PATH = f'{G_DRIVE_BASE}/sec_corpus_clean'
STUB_MANIFEST_PATH  = f'{G_DRIVE_BASE}/stub_files_manifest.txt'
OUTPUT_DIR          = f'{G_DRIVE_BASE}'  # where results CSVs will be saved

import os
assert os.path.exists(G_DRIVE_CORPUS_PATH), f"Corpus folder not found: {G_DRIVE_CORPUS_PATH}"
print(f"Corpus folder OK. Contains {len(os.listdir(G_DRIVE_CORPUS_PATH))} files.")

## 1. Build the stub exclusion list

Read `stub_files_manifest.txt` and build a set of files to skip during RAG ingestion and Challenger lookup.

**Also hard-coded:** JPM Item 8 FY2021–2022 (absent from corpus, confirmed by data team's Phase 7 smoke test).

In [ ]:
# ─── Build the exclusion set ──────────────────────────────────────────────────
EXCLUDED_FILES = set()

# Read from the manifest
if os.path.exists(STUB_MANIFEST_PATH):
    with open(STUB_MANIFEST_PATH, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#') or line.startswith('\\#'):
                continue
            # Each line starts with the filename (space-separated, first token)
            filename = line.split()[0]
            EXCLUDED_FILES.add(filename)
    print(f"Loaded {len(EXCLUDED_FILES)} stub filenames from manifest.")
else:
    print("Warning: stub manifest not found, continuing without it.")

# Also exclude JPM Item 8 FY2021-2022 (absent, not stubs)
# We can't list exact filenames since accession IDs are unknown for absent files,
# but we'll catch them at ingestion time via this ticker/year/section combo.
HARD_EXCLUDE_COMBOS = {
    ('JPM', '2021', 'Item8'),
    ('JPM', '2022', 'Item8'),
}

print(f"Total exclusion rules: {len(EXCLUDED_FILES)} stubs + {len(HARD_EXCLUDE_COMBOS)} absent file combos.")

## 2. Filename parser

Files are named like `AAPL_0000320193-22-000108_2022_Item1A.txt`.\
Format: `{TICKER}_{ACCESSION}_{FISCALYEAR}_{Section}.txt`

**Critical:** use the fiscal year embedded in the filename, **NOT** the accession year (mentioned in handoff file).

In [ ]:
import re

def parse_filename(filename):
    """
    Parse 'AAPL_0000320193-22-000108_2022_Item1A.txt' into structured metadata.
    Returns None if format doesn't match.
    """
    base = filename.replace('.txt', '')
    # Split on underscore: [TICKER, ACCESSION, FISCALYEAR, Section]
    parts = base.split('_')
    if len(parts) != 4:
        return None

    ticker, accession, fiscal_year, section_raw = parts

    # Normalise section: "Item1A" -> "Item 1A"
    section_normalized = section_raw.replace('Item', 'Item ')

    return {
        'ticker': ticker,
        'accession': accession,
        'fiscal_year': fiscal_year,
        'section_raw': section_raw,           # e.g. "Item1A" (matches Challenger's file lookup)
        'section': section_normalized,         # e.g. "Item 1A" (for display + RAG metadata)
        'filename': filename,
    }


def should_exclude(filename):
    """Returns True if this file should be skipped (stub or absent)."""
    if filename in EXCLUDED_FILES:
        return True
    parsed = parse_filename(filename)
    if parsed is None:
        return True
    combo = (parsed['ticker'], parsed['fiscal_year'], parsed['section_raw'])
    return combo in HARD_EXCLUDE_COMBOS


# Quick test
test_files = [
    'AAPL_0000320193-22-000108_2022_Item1A.txt',
    'JPM_xxx_2021_Item8.txt',  # should be excluded
]
for f in test_files:
    print(f"{f} -> parsed: {parse_filename(f)}, excluded: {should_exclude(f)}")

## 3. Load Qwen 2.5 7B (shared by both pipelines)

Loads in 4-bit quantization (~5GB VRAM on T4). Takes ~2 minutes the first time.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

print(f"Loading {MODEL_ID} in 4-bit quantization...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
challenger_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)
print("Qwen loaded.")

In [ ]:
import glob

# Default fiscal year for fetching a section
# (safe-for-numerical tickers are AAPL/MSFT/GOOGL)
DEFAULT_FISCAL_YEAR = '2022'


def find_section_file(ticker, section_raw, fiscal_year=DEFAULT_FISCAL_YEAR):
    """
    Find the file matching {TICKER}_*_{YEAR}_{Section}.txt.
    Returns the full path, or None if not found / excluded.

    Used by:
      - verify_golden_dataset() in Section 7.1 (ground truth verification)
    """
    pattern = f"{ticker}_*_{fiscal_year}_{section_raw}.txt"
    matches = glob.glob(os.path.join(G_DRIVE_CORPUS_PATH, pattern))
    matches = [m for m in matches if not should_exclude(os.path.basename(m))]
    if not matches:
        return None
    return matches[0]

In [ ]:
def generate_answer(prompt_text):
    """Generates an answer with Qwen for the Challenger pipeline."""
    messages = [
        {"role": "system", "content": "You are an expert financial analyst."},
        {"role": "user", "content": prompt_text}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(challenger_model.device)

    with torch.inference_mode():
        generated_ids = challenger_model.generate(
            **model_inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=True
        )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


print("generate_answer() ready.")

## 4. Challenger Pipeline (Full-Document Stuffing)

The Challenger concatenates all four sections (Item 1, Item 1A, Item 7, Item 8) of the target 10-K filing and feeds them as a single long-context prompt to Qwen 2.5-7B. A content-aware section localization step (via `section_recovery.py`) ensures structurally faithful section boundaries before concatenation.

The truncation budget is set to **8,000 words** (≈ 10K tokens), the practical input budget under our compute environment. \
In case If an OOM occurs on a smaller GPU, lower `MAX_FULL_DOC_WORDS`.

In [ ]:
# ─── Challenger Pipeline (Full-Document Baseline) ────────────────────────────
# Per the original proposal: "Long-context LLM with full document stuffing
# (no retrieval)" as the Challenger to the RAG Champion.

# Maximum word budget for the stuffed prompt. Qwen 2.5 supports 128K tokens
# (~96K words). We cap at 80K words to leave headroom for the system prompt,
# question, and generated answer.
MAX_FULL_DOC_WORDS = 8000


def assemble_full_doc_prompt(ticker, question, fiscal_year=DEFAULT_FISCAL_YEAR):
    """
    Builds a Challenger prompt by concatenating all available section files
    for the given ticker + fiscal year.
    """
    all_sections = []
    for section_raw in ['Item1', 'Item1A', 'Item7', 'Item8']:
        filepath = find_section_file(ticker, section_raw, fiscal_year)
        if filepath is None:
            continue  # skip stubs / absent files
        with open(filepath, 'r', encoding='utf-8') as f:
            section_text = f.read()
        all_sections.append(f"=== {section_raw} ===\n{section_text}")

    if not all_sections:
        return f"ERROR: No section files found for {ticker} FY{fiscal_year}."

    full_doc = "\n\n".join(all_sections)

    # Truncate if needed
    words = full_doc.split()
    if len(words) > MAX_FULL_DOC_WORDS:
        full_doc = " ".join(words[:MAX_FULL_DOC_WORDS]) + "\n...[TRUNCATED]"

    prompt = f"""You are an expert financial analyst. Answer the user's question using ONLY the provided SEC 10-K content. Do not use outside knowledge.

Company Ticker: {ticker}
Fiscal Year: {fiscal_year}

--- 10-K CONTENT (multiple sections concatenated) ---
{full_doc}
--- END CONTENT ---

Question: {question}
Answer:"""
    return prompt


# Smoke test
test_prompt = assemble_full_doc_prompt(
    "AAPL", "What was Apple's total net sales in fiscal year 2022?", "2022"
)
print(f"Full-doc prompt size: {len(test_prompt):,} chars")
print(f"First 400 chars:\n{test_prompt[:400]}")

In [ ]:
# Import section recovery: fixes a data-layer bug where the cleaned corpus
# has misaligned filenames (e.g. AAPL_2022_Item7.txt actually contains
# Item 1A content, while the real Item 7 sits inside ..._Item8.txt).
# We re-parse each raw file using regex on canonical SEC headers
# (e.g. "Item 7. Management's Discussion") to recover the true section
# boundaries before feeding the Challenger.
#
# Note: this fix only matters for the Challenger (Full-Doc), which trusts
# filenames when concatenating sections. The Champion (RAG) is unaffected
# because it chunks and retrieves section-agnostically.
import sys
sys.path.insert(0, 'data/SEC_NLP')
from section_recovery import assemble_recovered_full_doc

## 5. Champion Pipeline (RAG)

### 5.1 Build the Qdrant index

- Excludes stub files and known-absent files
- Parses ticker / fiscal_year / section from filename
- Uses 1000-char chunks with 150-char overlap (data team's Phase 7 recommendation for prose)
- Stores metadata for ticker+year filtering at retrieval time

**Runs in ~5-10 minutes** depending on corpus size.

In [ ]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue

print("Loading BGE-Small embedder...")
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
vector_size = embedder.get_sentence_embedding_dimension()

print("Initializing Qdrant (in-memory)...")
qdrant = QdrantClient(":memory:")
COLLECTION_NAME = "sec_filings"

if qdrant.collection_exists(COLLECTION_NAME):
    qdrant.delete_collection(COLLECTION_NAME)

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
)


def chunk_text(text, chunk_size=1000, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


print(f"\nIngesting from {G_DRIVE_CORPUS_PATH}...")
all_files = sorted(os.listdir(G_DRIVE_CORPUS_PATH))
skipped_count = 0
ingested_count = 0
global_point_id = 0

for filename in all_files:
    if not filename.endswith('.txt'):
        continue

    if should_exclude(filename):
        skipped_count += 1
        continue

    parsed = parse_filename(filename)
    if parsed is None:
        skipped_count += 1
        continue

    filepath = os.path.join(G_DRIVE_CORPUS_PATH, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        document_text = f.read()

    chunks = chunk_text(document_text)
    chunk_embeddings = embedder.encode(chunks, show_progress_bar=False)

    points = []
    for chunk, embedding in zip(chunks, chunk_embeddings):
        points.append(PointStruct(
            id=global_point_id,
            vector=embedding.tolist(),
            payload={
                "ticker":      parsed['ticker'],
                "fiscal_year": parsed['fiscal_year'],
                "section":     parsed['section'],
                "filename":    filename,
                "text":        chunk
            }
        ))
        global_point_id += 1

    qdrant.upsert(collection_name=COLLECTION_NAME, points=points)
    ingested_count += 1

print(f"\n✓ Ingestion complete.")
print(f"  Files ingested: {ingested_count}")
print(f"  Files skipped:  {skipped_count}")
print(f"  Total chunks:   {global_point_id}")

### 5.2 RAG retrieval (ticker-filtered) + generation

In [ ]:
def rag_retrieve(question, ticker, fiscal_year=None, top_k=5):
    """
    Retrieve top-k chunks for a question, filtered by ticker (and optionally fiscal year).
    Returns list of hits with .payload and .score.
    """
    q_embed = embedder.encode(question).tolist()

    must_conditions = [
        FieldCondition(key="ticker", match=MatchValue(value=ticker))
    ]
    if fiscal_year is not None:
        must_conditions.append(
            FieldCondition(key="fiscal_year", match=MatchValue(value=fiscal_year))
        )

    hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_embed,
        query_filter=Filter(must=must_conditions),
        limit=top_k
    ).points
    return hits


def build_rag_prompt(question, hits):
    """Assemble the RAG prompt from retrieved chunks."""
    retrieved_context = ""
    for i, hit in enumerate(hits):
        chunk = hit.payload['text'].strip()
        section = hit.payload.get('section', '?')
        year = hit.payload.get('fiscal_year', '?')
        retrieved_context += f"[Chunk {i+1}] ({section}, FY{year}):\n{chunk}\n\n"

    return f"""You are an expert financial analyst. Answer the user's question using ONLY the provided SEC filing excerpts.
The text may contain financial tables with irregular formatting — read numbers carefully.
If the answer is not contained in the excerpts, you must state "The provided text does not contain the answer."
Do not use outside knowledge.

--- RETRIEVED SEC EXCERPTS ---
{retrieved_context}
--- END EXCERPTS ---

Question: {question}
Answer:"""


def generate_rag_answer(prompt_text):
    """Generates an answer with Qwen for the Champion pipeline."""
    messages = [
        {"role": "system", "content": "You are a precise, grounded financial AI."},
        {"role": "user", "content": prompt_text}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(challenger_model.device)

    with torch.inference_mode():
        generated_ids = challenger_model.generate(
            **model_inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=True
        )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


# Smoke test: retrieve + generate one answer
test_q = "What are the main cybersecurity threats to Apple's supply chain?"
test_hits = rag_retrieve(test_q, ticker="AAPL", fiscal_year=DEFAULT_FISCAL_YEAR, top_k=3)
print(f"Retrieved {len(test_hits)} hits:")
for i, h in enumerate(test_hits):
    print(f"  [{i+1}] {h.payload['section']} FY{h.payload['fiscal_year']} score={h.score:.3f}")

test_rag_prompt = build_rag_prompt(test_q, test_hits)
print("\n--- RAG answer ---")
print(generate_rag_answer(test_rag_prompt))

## 6. Independent Judge: GPT 4o

We do NOT let Qwen judge itself (self-judging bias is well documented).

**Paste your Anthropic API key below.** Get one free at https://console.anthropic.com.

In [ ]:
from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
# Install OpenAI SDK (only need once per session)
!pip install -q openai

from openai import OpenAI
import json, re, os

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

# Pick your judge model.
# gpt-4o-mini: cheap (~$0.05 for the 32 calls), fast, plenty good for this rubric
# gpt-4o:      ~10x more expensive but more reliable on edge cases
JUDGE_MODEL = "gpt-4o"


def judge_with_gpt(question, context, generated_answer, ground_truth):
    """
    Scores an answer on relevance and faithfulness using OpenAI.
    Function name kept as `judge_with_gpt` so the rest of the notebook works unchanged.
    Returns dict with normalized 0.0-1.0 scores + reasoning.
    """
    judge_prompt = f"""You are an expert evaluator assessing AI-generated answers to financial questions.

Evaluate the answer below on TWO dimensions. Return your scores as a JSON object.

QUESTION: {question}

CONTEXT PROVIDED TO THE AI (the only text it was allowed to use):
{context[:2000]}  [← truncated to 2000 chars for this evaluation call]

REFERENCE GROUND TRUTH (the known correct answer):
{ground_truth}

AI-GENERATED ANSWER:
{generated_answer}

SCORING RUBRIC:

answer_relevance (1-5):
  5 = Directly and completely answers the question with no irrelevant content
  4 = Good answer with minor irrelevant details
  3 = Partially answers — key constraint missed or answer is incomplete
  2 = Mostly off-topic
  1 = Completely irrelevant or refuses to answer ("not found")

faithfulness (1-5):
  5 = Every claim is directly supported by the provided context; no hallucination
  4 = Mostly grounded; one minor inference beyond the text
  3 = Some claims go beyond the context but aren't contradicted by it
  2 = Several claims that cannot be verified from the context
  1 = Significant fabrication or claims that contradict the context

Return ONLY valid JSON, no other text:
{{"answer_relevance": <int 1-5>, "faithfulness": <int 1-5>, "reasoning": "<one sentence>"}}"""

    try:
        response = client.chat.completions.create(
            model=JUDGE_MODEL,
            max_tokens=150,
            temperature=0,
            response_format={"type": "json_object"},  # forces valid JSON
            messages=[{"role": "user", "content": judge_prompt}]
        )
        raw = response.choices[0].message.content.strip()
        parsed = json.loads(raw)
        return {
            "answer_relevance": (parsed["answer_relevance"] - 1) / 4.0,
            "faithfulness":     (parsed["faithfulness"]     - 1) / 4.0,
            "reasoning":        parsed.get("reasoning", "")
        }
    except Exception as e:
        print(f"  Judge call failed: {e}")
        return {"answer_relevance": 0.0, "faithfulness": 0.0, "reasoning": f"ERROR: {e}"}


# Smoke test
test_scores = judge_with_gpt(
    question="What is Apple's main product?",
    context="Apple designs the iPhone, Mac, iPad, and wearables.",
    generated_answer="Apple's main product is the iPhone.",
    ground_truth="The iPhone is Apple's primary product line."
)
print("Smoke test:", test_scores)

In [ ]:
import torch
print(f"Total GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"Free memory:      {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")
print(f"Allocated:        {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved:         {torch.cuda.memory_reserved() / 1e9:.2f} GB")

## 8. Numerical Exact Match (for factual_lookup questions)

In [ ]:
def extract_numbers(text):
    """Returns set of numeric tokens. Handles '178,353' -> '178353'."""
    cleaned = re.sub(r'(\d),(\d)', r'\1\2', text)
    return set(re.findall(r'\d+\.?\d*', cleaned))


def numerical_exact_match(generated_answer, ground_truth):
    """
    Returns fraction of key numbers from ground_truth that appear in generated_answer.
    Returns None if ground_truth has no significant numbers (skip EM for that row).
    Filters out years (4-digit numbers starting with 20XX).
    """
    gt_numbers = extract_numbers(ground_truth)
    key_numbers = {
        n for n in gt_numbers
        if len(n.replace('.', '')) >= 4 and not re.match(r'^20\d{2}$', n)
    }
    if not key_numbers:
        return None
    answer_numbers = extract_numbers(generated_answer)
    found = key_numbers & answer_numbers
    return len(found) / len(key_numbers)


# Smoke test
em = numerical_exact_match(
    "Americas net sales were $178353 million in 2022 and $153306 million in 2021.",
    "$178,353 million in 2022 and $153,306 million in 2021."
)
print(f"EM test (should be 1.0): {em}")

In [ ]:
# ─── End-to-end Smoke Test ────────────────────────────────────────────────────
# Runs ONE full question through both pipelines + GPT-4o judge + EM.
# Validates the entire chain before you commit to filling 16 ground truths.

import time

# Use a real AAPL FY2022 Item 8 question with a known answer.
# (Item 8 is the strictest test: tables, numbers, parser quirks.)
smoke_test = {
    "question":     "What was Apple's total net sales in fiscal year 2022?",
    "ticker":       "AAPL",
    "fiscal_year":  "2022",
    "ground_truth": "Total net sales of $394,328 million in 2022",  # AAPL 10-K FY2022 actual figure
    "question_type": "factual_lookup",
}

print("="*70)
print("  END-TO-END SMOKE TEST")
print("="*70)
print(f"Question: {smoke_test['question']}")
print(f"Ground truth: {smoke_test['ground_truth']}")
print()

# ─── CHALLENGER ──────────────────────────────────────
print("─── CHALLENGER (Full-Document Stuffing + Qwen) ───────────────────────────")
t0 = time.time()

challenger_prompt = assemble_recovered_full_doc(
    smoke_test['ticker'], smoke_test['question'],
    fiscal_year=smoke_test['fiscal_year'],
    corpus_path=G_DRIVE_CORPUS_PATH,
    max_words=MAX_FULL_DOC_WORDS,
    exclude_fn=should_exclude,
)
def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))
challenger_tokens = count_tokens(challenger_prompt)
print(f"  Prompt size: {challenger_tokens} tokens ({len(challenger_prompt):,} chars)")

challenger_answer = generate_answer(challenger_prompt)
challenger_latency = time.time() - t0
print(f"  Latency: {challenger_latency:.1f}s")
print(f"  Answer: {challenger_answer[:300]}{'...' if len(challenger_answer) > 300 else ''}")

# Extract context for the judge — match new prompt template's delimiters
import re
ctx_match = re.search(r'--- 10-K CONTENT.*?---(.+?)--- END CONTENT ---',
                      challenger_prompt, re.DOTALL)
challenger_context = ctx_match.group(1).strip() if ctx_match else challenger_prompt

# ─── CHAMPION ─────────────────────────────────────────────────────────────────
print("\n─── CHAMPION (RAG + Qwen) ────────────────────────────────────────────────")
t0 = time.time()
hits = rag_retrieve(
    smoke_test['question'],
    ticker=smoke_test['ticker'],
    fiscal_year=smoke_test['fiscal_year'],
    top_k=5
)
print(f"  Retrieved {len(hits)} chunks:")
for i, h in enumerate(hits):
    print(f"    [{i+1}] {h.payload['section']} (score={h.score:.3f})")

champion_prompt = build_rag_prompt(smoke_test['question'], hits)
champion_tokens = count_tokens(champion_prompt)
print(f"  Prompt size: {champion_tokens} tokens")

champion_answer = generate_rag_answer(champion_prompt)
champion_latency = time.time() - t0
print(f"  Latency: {champion_latency:.1f}s")
print(f"  Answer: {champion_answer[:300]}{'...' if len(champion_answer) > 300 else ''}")

# Extract context for the judge
champion_context = "\n\n".join([h.payload.get('text', '') for h in hits])

# ─── JUDGE + EM ────────────────────────────────────────────────────────────────
print("\n─── EVALUATION ───────────────────────────────────────────────────────────")
challenger_scores = judge_with_gpt(
    smoke_test['question'], challenger_context, challenger_answer, smoke_test['ground_truth']
)
champion_scores = judge_with_gpt(
    smoke_test['question'], champion_context, champion_answer, smoke_test['ground_truth']
)
challenger_em = numerical_exact_match(challenger_answer, smoke_test['ground_truth'])
champion_em = numerical_exact_match(champion_answer, smoke_test['ground_truth'])

print(f"  Challenger: relevance={challenger_scores['answer_relevance']:.2f}  "
      f"faithfulness={challenger_scores['faithfulness']:.2f}  EM={challenger_em:.2f}")
print(f"  Champion:   relevance={champion_scores['answer_relevance']:.2f}  "
      f"faithfulness={champion_scores['faithfulness']:.2f}  EM={champion_em:.2f}")
print("\n✓ Smoke test complete. Ready to run full batch evaluation.")


## 7. Golden Dataset (16 questions)

⚠️ **You need to fill in the `FILL_IN_FROM_CORPUS` ground truths.** Open the corresponding `.txt` file in `sec_corpus_clean/` and copy the actual answer.

**Critical:** the parser strips `$` and `,` from numbers. Your ground truth must match the format **as stored in the .txt**, not the original EDGAR formatting. Run `verify_golden_dataset()` below to catch mismatches.

In [ ]:
golden_dataset = [
    # ─── AAPL FY2022 Item 1A (Risk Factors) ──────────────────────────────────
    {
        "test_id": 1, "ticker": "AAPL", "fiscal_year": "2022",
        "section_hint": "Item 1A", "question_type": "narrative",
        "question": "What are the main cybersecurity threats described in Apple's risk factors?",
        "ground_truth": "Apple identifies cybersecurity risks including ransomware and other cybersecurity attacks, unauthorized access to or releases of confidential information (including personal information), hacking, computer viruses, physical or electronic break-ins, malicious attacks, and attempts by hackers (including state-sponsored actors) to compromise the confidentiality, integrity or availability of systems. The Company notes attacks are expected to accelerate in frequency and sophistication, and that it is at relatively greater risk due to its high profile and the value of its confidential information.",
    },
    {
        "test_id": 2, "ticker": "AAPL", "fiscal_year": "2022",
        "section_hint": "Item 1A", "question_type": "narrative",
        "question": "What macroeconomic factors does Apple identify as risks to its business?",
        "ground_truth": "Apple identifies adverse macroeconomic conditions including inflation, slower growth or recession, new or increased tariffs and other barriers to trade, changes to fiscal and monetary policy, tighter credit, higher interest rates, high unemployment, and currency fluctuations. These can adversely impact consumer confidence and spending and materially adversely affect demand for the Company's products and services. Apple also cites financial market volatility, declines in income or asset values, energy shortages, and labor and healthcare costs.",
    },
    # ─── AAPL FY2022 Item 7 (MD&A) ───────────────────────────────────────────
    {
        "test_id": 3, "ticker": "AAPL", "fiscal_year": "2022",
        "section_hint": "Item 7", "question_type": "narrative",
        "question": "How does Apple's management explain the change in gross margin year over year?",
        "ground_truth": "Apple's total gross margin increased to $170,782 million in 2022 from $152,836 million in 2021, with the gross margin percentage rising from 41.8% to 43.3%. Management attributes the increase in Products gross margin to a different Products mix and higher Products volume, partially offset by weakness in foreign currencies relative to the U.S. dollar. Services gross margin increased due to higher Services net sales, partially offset by foreign currency weakness; Services gross margin percentage increased due to improved leverage and a different Services mix.",
    },
    {
        "test_id": 4, "ticker": "AAPL", "fiscal_year": "2022",
        "section_hint": "Item 7", "question_type": "narrative",
        "question": "What does Apple's management say about its Services segment growth strategy?",
        "ground_truth": "Apple's Services net sales increased during 2022 compared to 2021 due primarily to higher net sales from advertising, cloud services, and the App Store. Services net sales reached $78,129 million in 2022 (up 14% from $68,425 million in 2021). Services include advertising, AppleCare, cloud, digital content, payment and other services.",
    },
    # ─── AAPL FY2022 Item 8 (Financial Statements) ───────────────────────────
    {
        "test_id": 5, "ticker": "AAPL", "fiscal_year": "2022",
        "section_hint": "Item 8", "question_type": "factual_lookup",
        "question": "What were Apple's net sales for the Americas segment in fiscal year 2022?",
        "ground_truth": "Americas net sales were $169,658 million in 2022, compared to $153,306 million in 2021 (an increase of 11%). Americas net sales increased during 2022 compared to 2021 due primarily to higher net sales of iPhone, Services and Mac.",
    },
    {
        "test_id": 6, "ticker": "AAPL", "fiscal_year": "2022",
        "section_hint": "Item 8", "question_type": "factual_lookup",
        "question": "What was Apple's total net sales in fiscal year 2022?",
        "ground_truth": "Apple's total net sales were $394,328 million in 2022, compared to $365,817 million in 2021 (an increase of 8% or $28.5 billion). The increase was driven primarily by higher net sales of iPhone, Services and Mac.",
    },
    # ─── AAPL FY2022 Item 1 (Business) ───────────────────────────────────────
    {
        "test_id": 7, "ticker": "AAPL", "fiscal_year": "2022",
        "section_hint": "Item 1", "question_type": "narrative",
        "question": "How does Apple describe its main product categories in its business overview?",
        "ground_truth": "Apple's net sales are organized into five product categories: iPhone ($205,489 million, 7% growth in 2022), Mac ($40,177 million, 14% growth), iPad ($29,292 million, down 8%), Wearables, Home and Accessories ($41,241 million, 7% growth — including AirPods, Apple TV, Apple Watch, Beats products, HomePod mini), and Services ($78,129 million, 14% growth — including advertising, AppleCare, cloud, digital content, payment and other services).",
    },
    # ─── AAPL Cross-Section ────────────────────────────────────────────────
    {
        "test_id": 8, "ticker": "AAPL", "fiscal_year": "2022",
        "section_hint": None, "question_type": "cross_section",
        "question": "Apple identifies supply chain disruption as a risk — did this risk materialize in its financial results?",
        "ground_truth": "Apple identifies supply chain disruption in Item 1A as a major risk, citing dependence on outsourcing partners primarily in Asia, single-source suppliers, COVID-19-related disruptions, and component shortages including the global semiconductor industry shortage. Despite these risks, Apple's FY2022 financial results showed total net sales increased 8% to $394,328 million (up $28.5 billion from $365,817 million in 2021), with gross margin increasing to $170,782 million (43.3% margin, up from 41.8%). The weakness in foreign currencies relative to the U.S. dollar had an unfavorable year-over-year impact on net sales, but the supply chain risks did not prevent overall revenue and margin growth.",
    },
    # ─── MSFT FY2022 Item 1A (Risk Factors) ──────────────────────────────────
    {
        "test_id": 9, "ticker": "MSFT", "fiscal_year": "2022",
        "section_hint": "Item 1A", "question_type": "narrative",
        "question": "What cybersecurity and AI-related risks does Microsoft disclose in its risk factors?",
        "ground_truth": "Microsoft identifies cyberattacks and security vulnerabilities that could lead to reduced revenue, increased costs, liability claims, or harm to reputation. Threats include attacks by individual hackers, sophisticated organizations, state-sponsored actors and nation-states using malware, vulnerability exploits, social engineering, denial of service attacks, and supply chain attacks (citing the Solorigate/Nobelium incident in late 2020). Microsoft notes cyberthreats are constantly evolving, may persist in environments for long periods, and that nation-state attacks may intensify during periods of armed conflict such as the ongoing conflict in Ukraine. AI-related risks include emerging legislative and regulatory action in AI and content moderation areas that may increase costs.",
    },
    {
        "test_id": 10, "ticker": "MSFT", "fiscal_year": "2022",
        "section_hint": "Item 1A", "question_type": "narrative",
        "question": "What regulatory risks does Microsoft identify relating to antitrust or government oversight?",
        "ground_truth": "Microsoft notes that government regulatory authorities and private litigants scrutinize its business under U.S. and foreign competition laws. The European Commission closely scrutinizes the design of high-volume Microsoft products and terms on which technologies (file formats, programming interfaces, protocols) are made available to other companies. The Commission previously ordered Microsoft to create Windows versions without certain multimedia technologies (2004) and accepted commitments addressing web browsing software competition concerns (2009, expired 2014). Remaining obligations may limit ability to innovate in Windows or other products.",
    },
    # ─── MSFT FY2022 Item 7 (MD&A) ───────────────────────────────────────────
    {
        "test_id": 11, "ticker": "MSFT", "fiscal_year": "2022",
        "section_hint": "Item 7", "question_type": "narrative",
        "question": "How does Microsoft explain the revenue growth in its Intelligent Cloud segment?",
        "ground_truth": "Microsoft's Intelligent Cloud segment revenue increased driven by Azure and other cloud services. The segment includes server products and cloud services (Azure, SQL Server, Windows Server, Visual Studio, System Center, Nuance, GitHub) and Enterprise Services. Azure revenue growth is mainly affected by infrastructure-as-a-service and platform-as-a-service consumption-based services, and per user-based services such as Enterprise Mobility + Security. The Microsoft Cloud as a whole drove cost of revenue growth of $10.4 billion (20%).",
    },
    {
        "test_id": 12, "ticker": "MSFT", "fiscal_year": "2022",
        "section_hint": "Item 7", "question_type": "narrative",
        "question": "What capital expenditure commitments does Microsoft's management discussion mention?",
        "ground_truth": "Microsoft states it expects capital expenditures to increase in coming years to support growth in its cloud offerings. The Company has operating and finance leases for datacenters, corporate offices, research and development facilities, Microsoft Experience Centers, and certain equipment. Cash, cash equivalents and short-term investments totaled $104.8 billion as of June 30, 2022 (down from $130.3 billion in 2021), and management believes available liquidity is sufficient for capital expenditures, dividends, share repurchases, debt maturities, and the TCJA transition tax for at least the next 12 months.",
    },
    # ─── MSFT FY2022 Item 8 (Financial Statements) ───────────────────────────
    {
        "test_id": 13, "ticker": "MSFT", "fiscal_year": "2022",
        "section_hint": "Item 8", "question_type": "factual_lookup",
        "question": "What was Microsoft's total revenue for fiscal year 2022?",
        "ground_truth": "Microsoft's total revenue was $198,270 million in fiscal year 2022, compared to $168,088 million in fiscal year 2021 — an 18% increase.",
    },
    {
        "test_id": 14, "ticker": "MSFT", "fiscal_year": "2022",
        "section_hint": "Item 8", "question_type": "factual_lookup",
        "question": "What was Microsoft's net income for fiscal year 2022?",
        "ground_truth": "Microsoft's net income was $72,738 million in fiscal year 2022, compared to $61,271 million in fiscal year 2021 — a 19% increase. Operating income was $83,383 million (up 19%) and diluted earnings per share was $9.65 (up 20%).",
    },
    # ─── MSFT FY2022 Item 1 (Business) ───────────────────────────────────────
    {
        "test_id": 15, "ticker": "MSFT", "fiscal_year": "2022",
        "section_hint": "Item 1", "question_type": "narrative",
        "question": "How does Microsoft describe its three main business segments?",
        "ground_truth": "Microsoft's three reportable segments are: (1) Productivity and Business Processes, consisting of Office Commercial, Office Consumer, LinkedIn (Talent Solutions, Marketing Solutions, Premium Subscriptions, Sales Solutions), and Dynamics business solutions; (2) Intelligent Cloud, consisting of Server products and cloud services including Azure, SQL Server, Windows Server, Visual Studio, Nuance, GitHub, and Enterprise Services; and (3) More Personal Computing, including Windows, Search and news advertising, gaming, and devices.",
    },
    # ─── MSFT Cross-Section ──────────────────────────────────────────────────
    {
        "test_id": 16, "ticker": "MSFT", "fiscal_year": "2022",
        "section_hint": None, "question_type": "cross_section",
        "question": "Microsoft cites AI investment as both a growth driver and a risk — how does the financial data support or complicate this claim?",
        "ground_truth": "Microsoft positions AI as a strategic investment area: Azure incorporates AI, IoT, cognitive services and machine learning capabilities, and AI is a competitive differentiator in cybersecurity solutions. On the risk side, Microsoft identifies emerging legislative and regulatory action in AI and content moderation that may increase costs. The financial data shows Intelligent Cloud (the segment most associated with AI/cloud investment) drove FY2022 revenue growth, with total revenue increasing 18% to $198,270 million and net income up 19% to $72,738 million. Microsoft Cloud gross margin grew, and management expects capital expenditures to increase to support cloud growth — consistent with AI/cloud being a profitable growth driver despite regulatory headwinds.",
    },
]

# Sanity check
unfilled = [r['test_id'] for r in golden_dataset if r['ground_truth'] == 'FILL_IN_FROM_CORPUS']
if unfilled:
    print(f"⚠ {len(unfilled)} ground truths still need to be filled in: IDs {unfilled}")
else:
    print(f"✓ Golden dataset ready: {len(golden_dataset)} questions")

### 7.1 Verify ground truths exist in the corpus

In [ ]:
def verify_golden_dataset(dataset):
    """
    For each filled ground_truth, check that a key phrase appears in the corresponding .txt.
    Catches the 'parser stripped formatting' bug.
    """
    print("Verifying ground truths against corpus files...\n")
    issues = []

    for row in dataset:
        if row['ground_truth'] == 'FILL_IN_FROM_CORPUS':
            continue
        if row['section_hint'] is None:
            continue  # cross-section spans multiple files; skip auto-check

        safe_item = row['section_hint'].replace(' ', '')
        filepath = find_section_file(row['ticker'], safe_item, row['fiscal_year'])

        if filepath is None:
            issues.append(f"ID {row['test_id']}: file not found for "
                          f"{row['ticker']} {row['section_hint']} FY{row['fiscal_year']}")
            continue

        with open(filepath, 'r', encoding='utf-8') as f:
            corpus_text = f.read().lower()

        # Use first ~40 chars of ground truth as a search key,
        # stripping $ and , which the parser may have removed
        key_phrase = re.sub(r'[\$,]', '', row['ground_truth']).lower()[:40].strip()

        if key_phrase not in corpus_text:
            issues.append(
                f"ID {row['test_id']}: key phrase not found in {os.path.basename(filepath)}\n"
                f"  Searched for: '{key_phrase}'"
            )
        else:
            print(f"  ✓ ID {row['test_id']}: verified")

    if issues:
        print(f"\n⚠ {len(issues)} issue(s):")
        for i in issues:
            print(f"  {i}")
        print("\nFor failed cases, open the .txt and update ground_truth to match exactly.")
    else:
        print("\n✓ All filled ground truths verified.")


verify_golden_dataset(golden_dataset)

## 9. Batch Evaluation Runner

For each question, runs BOTH pipelines, judges each, computes EM. Logs latency and token counts.

**Expect ~5-10 minutes** for 16 questions (mostly Qwen generation time on T4).

In [ ]:
# revised batch runner
import time, gc, pandas as pd, re

def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

results = []
print(f"Running batch evaluation on {len(golden_dataset)} questions...")
print("="*70)

for idx, item in enumerate(golden_dataset):
    test_id = item['test_id']
    q = item['question']
    ticker = item['ticker']
    fy = item['fiscal_year']
    q_type = item['question_type']
    gt = item['ground_truth']

    if gt == "FILL_IN_FROM_CORPUS":
        print(f"\n[{idx+1}/{len(golden_dataset)}] ID {test_id}: SKIPPED (ground truth not filled)")
        continue

    print(f"\n[{idx+1}/{len(golden_dataset)}] ID {test_id} ({ticker} FY{fy}, {q_type})")
    print(f"  Q: {q[:80]}{'...' if len(q) > 80 else ''}")

    result = {
        "test_id": test_id, "ticker": ticker, "fiscal_year": fy,
        "question_type": q_type, "question": q, "ground_truth": gt,
    }

    # ─── CHALLENGER: Full-Document Stuffing + Qwen ───────────────────────
    try:
        t0 = time.time()
        challenger_prompt = assemble_recovered_full_doc(
            ticker, q, fiscal_year=fy,
            corpus_path=G_DRIVE_CORPUS_PATH,
            max_words=MAX_FULL_DOC_WORDS,
            exclude_fn=should_exclude,
        )
        challenger_tokens_in = count_tokens(challenger_prompt)
        challenger_answer = generate_answer(challenger_prompt)
        challenger_latency = time.time() - t0
        ctx_match = re.search(r'--- 10-K CONTENT.*?---(.+?)--- END CONTENT ---',
                              challenger_prompt, re.DOTALL)
        challenger_context = ctx_match.group(1).strip() if ctx_match else challenger_prompt
        print(f"  Challenger → full doc | {challenger_tokens_in} tokens | {challenger_latency:.1f}s")
    except Exception as e:
        print(f"  Challenger FAILED: {e}")
        result.update({
            "challenger_answer": f"ERROR: {e}",
            "challenger_tokens_in": 0, "challenger_latency_s": 0,
            "challenger_relevance": 0, "challenger_faithfulness": 0,
            "challenger_em": 0, "challenger_judge_reasoning": "error",
        })
        continue

    # ─── CHAMPION: RAG + Qwen ─────────────────────────────────────────────
    try:
        t0 = time.time()
        hits = rag_retrieve(q, ticker=ticker, fiscal_year=fy, top_k=5)
        sections_retrieved = [h.payload.get('section', '?') for h in hits]
        champion_prompt = build_rag_prompt(q, hits)
        champion_tokens_in = count_tokens(champion_prompt)
        champion_answer = generate_rag_answer(champion_prompt)
        champion_latency = time.time() - t0
        champion_context = "\n\n".join([h.payload.get('text', '') for h in hits])
        print(f"  Champion   → RAG ({len(hits)} chunks) | {champion_tokens_in} tokens | {champion_latency:.1f}s")
    except Exception as e:
        print(f"  Champion FAILED: {e}")
        result.update({
            "champion_answer": f"ERROR: {e}",
            "champion_tokens_in": 0, "champion_latency_s": 0,
            "champion_relevance": 0, "champion_faithfulness": 0,
            "champion_em": 0, "champion_judge_reasoning": "error",
        })
        continue

    # ─── JUDGE with GPT-4o ────────────────────────────────────────────────
    challenger_scores = judge_with_gpt(q, challenger_context, challenger_answer, gt)
    time.sleep(0.5)
    champion_scores = judge_with_gpt(q, champion_context, champion_answer, gt)

    # ─── EM ───────────────────────────────────────────────────────────────
    if q_type == 'factual_lookup':
        challenger_em = numerical_exact_match(challenger_answer, gt)
        champion_em = numerical_exact_match(champion_answer, gt)
    else:
        challenger_em = None
        champion_em = None

    result.update({
        "challenger_answer": challenger_answer,
        "challenger_tokens_in": challenger_tokens_in,
        "challenger_latency_s": round(challenger_latency, 2),
        "challenger_relevance": round(challenger_scores['answer_relevance'], 3),
        "challenger_faithfulness": round(challenger_scores['faithfulness'], 3),
        "challenger_em": challenger_em,
        "challenger_judge_reasoning": challenger_scores['reasoning'],

        "champion_sections_retrieved": str(sections_retrieved),
        "champion_answer": champion_answer,
        "champion_tokens_in": champion_tokens_in,
        "champion_latency_s": round(champion_latency, 2),
        "champion_relevance": round(champion_scores['answer_relevance'], 3),
        "champion_faithfulness": round(champion_scores['faithfulness'], 3),
        "champion_em": champion_em,
        "champion_judge_reasoning": champion_scores['reasoning'],
    })

    results.append(result)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_results = pd.DataFrame(results)
print("\n" + "="*70)
print(f"Batch evaluation complete: {len(df_results)} questions evaluated.")
print("="*70)


## 10. Results summary

In [ ]:
print("="*70)
print("  PROBLEM 2 EVALUATION SUMMARY")
print("="*70)

metric_cols = [
    ('Relevance',     'challenger_relevance',     'champion_relevance'),
    ('Faithfulness',  'challenger_faithfulness',  'champion_faithfulness'),
    ('Latency (s)',   'challenger_latency_s',     'champion_latency_s'),
    ('Tokens In',     'challenger_tokens_in',     'champion_tokens_in'),
]

print(f"\n{'Metric':<20} {'Challenger':>15} {'Champion':>15} {'Winner':>12}")
print("-" * 65)

for label, ch_col, champ_col in metric_cols:
    ch_val    = df_results[ch_col].mean()
    champ_val = df_results[champ_col].mean()
    if label in ('Latency (s)', 'Tokens In'):
        winner = 'Challenger' if ch_val < champ_val else 'Champion'
    else:
        winner = 'Challenger' if ch_val > champ_val else 'Champion'
    print(f"{label:<20} {ch_val:>15.3f} {champ_val:>15.3f} {winner:>12}")

# Exact Match (factual_lookup only)
em_rows = df_results[df_results['challenger_em'].notna()]
if not em_rows.empty:
    ch_em    = em_rows['challenger_em'].mean()
    champ_em = em_rows['champion_em'].mean()
    winner = 'Challenger' if ch_em > champ_em else 'Champion'
    print(f"{'Exact Match (num.)':<20} {ch_em:>15.3f} {champ_em:>15.3f} {winner:>12}")
    print(f"  (on {len(em_rows)} factual_lookup questions)")

# Breakdown by question type
print("\nRelevance by question type:")
print(f"  {'Type':<20} {'Challenger':>12} {'Champion':>12} {'Δ (Champ-Chal)':>16}")
print("  " + "-" * 60)
for q_type in ['factual_lookup', 'narrative', 'cross_section']:
    sub = df_results[df_results['question_type'] == q_type]
    if sub.empty:
        continue
    ch    = sub['challenger_relevance'].mean()
    champ = sub['champion_relevance'].mean()
    delta = champ - ch
    arrow = '↑' if delta > 0.05 else ('↓' if delta < -0.05 else '≈')
    print(f"  {q_type:<20} {ch:>12.3f} {champ:>12.3f} {delta:>+12.3f}  {arrow}")

# Cross-section detail
cross = df_results[df_results['question_type'] == 'cross_section']
if not cross.empty:
    print("\nCross-section questions:")
    for _, r in cross.iterrows():
        print(f"  ID {r['test_id']}: Challenger rel={r['challenger_relevance']:.2f} "
              f"vs Champion={r['champion_relevance']:.2f}")

## 12. Save results

In [ ]:
display_cols = [
    'test_id', 'ticker', 'fiscal_year', 'question_type',
    'challenger_relevance', 'challenger_faithfulness', 'challenger_em',
    'champion_sections_retrieved', 'champion_relevance', 'champion_faithfulness', 'champion_em',
]
pd.set_option('display.max_colwidth', 40)
display(df_results[display_cols])

full_path = f'{OUTPUT_DIR}/evaluation_results_full.csv'
df_results.to_csv(full_path, index=False, encoding='utf-8-sig')
print(f"\nFull results saved to: {full_path}")

summary_path = f'{OUTPUT_DIR}/evaluation_results_summary.csv'
df_results[display_cols].to_csv(summary_path, index=False, encoding='utf-8-sig')
print(f"Summary saved to: {summary_path}")


## Section 13: Interactive QA Demo

In [ ]:
# ─── Section 13: Interactive QA Demo ─────────────────────────────────────
# A simple wrapper showing how the Champion (RAG) system answers arbitrary
# questions over the SEC 10-K corpus. For demo and slide screenshots.

def ask(question, ticker, fiscal_year="2022", show_chunks=True):
    """
    Interactive QA over SEC 10-K filings using the Champion (RAG) pipeline.
    """
    print(f"📊 Question: {question}")
    print(f"   Company: {ticker} | Fiscal Year: {fiscal_year}\n")

    # Retrieve
    hits = rag_retrieve(question, ticker=ticker, fiscal_year=fiscal_year, top_k=5)
    if not hits:
        print(f"⚠️  No relevant content found for {ticker} FY{fiscal_year}.")
        return

    if show_chunks:
        print(f"🔍 Retrieved {len(hits)} chunks:")
        for i, h in enumerate(hits):
            print(f"   [{i+1}] {h.payload['section']} (score={h.score:.3f})")
        print()

    # Generate
    prompt = build_rag_prompt(question, hits)
    answer = generate_rag_answer(prompt)

    print(f"💬 Answer:\n{answer}\n")
    return answer


# ─── Demo: 7 questions across 3 companies (AAPL/MSFT/GOOGL) × 3 years (2020-2022) ──
print("="*70)
print("  QA SYSTEM DEMO — SEC 10-K Question Answering with RAG")
print("="*70)

demo_questions = [
    # ─── AAPL（2020, 2021, 2022）─────────────────────────────
    ("How did COVID-19 affect Apple's business operations in fiscal year 2020?",
     "AAPL", "2020"),
    ("What was Apple's iPhone revenue in fiscal year 2021?",
     "AAPL", "2021"),
    ("How much did Apple spend on share repurchases and dividends in fiscal year 2022?",
     "AAPL", "2022"),

    # ─── MSFT（2020, 2022）──────────────────────────────────
    ("How many members did LinkedIn have at the end of fiscal year 2020?",
     "MSFT", "2020"),
    ("What was the purchase price of Microsoft's Nuance acquisition?",
     "MSFT", "2022"),

    # ─── GOOGL FY2022 ──────────────────────────────────────
    ("What was Alphabet's total revenue in 2022?",
     "GOOGL", "2022"),
    ("How much did Alphabet spend on share repurchases in 2022?",
     "GOOGL", "2022"),
]

for q, ticker, fy in demo_questions:
    print("\n" + "─"*70)
    ask(q, ticker, fy, show_chunks=True)